# ModelCompression — Colab quick start
Choose **Runtime → Change runtime type → GPU**, fill in the repository URL, and run all cells. The defaults are deliberately small enough for a smoke test.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/model-compression.git" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}
if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Set REPO_URL to your published GitHub repository first.")
!git clone --depth 1 --branch "{BRANCH}" "{REPO_URL}" /content/model-compression
%cd /content/model-compression
%pip install -q -e .

In [ ]:
MODEL = "qwen3-0.6b" # @param ["qwen3-0.6b", "qwen3-1.7b", "qwen2.5-0.5b"]
DATASET = "mquake-cf3k" # @param ["mquake-cf3k", "mquake-cf9k", "mquake-openmath", "openmath", "2wiki"]
LIMIT = 3 # @param {type:"integer"}
CUSTOM_MODEL_REVISION = "" # @param {type:"string"}
print(f"Model: {MODEL}\nDataset: {DATASET}\nExamples: {LIMIT}")

## Verify generation
This downloads the selected model on the first run. Set `MODEL` to any compatible Hugging Face repository ID if you do not want a preset.

In [ ]:
import subprocess
common = ["--model", MODEL, "--device", "cuda"]
if CUSTOM_MODEL_REVISION:
    common += ["--revision", CUSTOM_MODEL_REVISION]
subprocess.run(["model-compression", "generate", *common, "--prompt", "What is 2 + 2? Answer briefly."], check=True)

## Score MLP channels
This computes a simple mean absolute answer-loss gate gradient. Start with 1–3 examples; use `LIMIT = 0` only when you intend to score the complete selected file.

In [ ]:
subprocess.run([
    "model-compression", "score", *common,
    "--dataset", DATASET, "--limit", str(LIMIT),
    "--output", "outputs/channel_scores.pt",
], check=True)
import torch
result = torch.load("outputs/channel_scores.pt", map_location="cpu", weights_only=False)
print({key: value for key, value in result.items() if key != "scores"})
print("score tensor shape:", tuple(result["scores"].shape))